# XAI: LIME

## 1. Imports

In [ ]:
import os
import json
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import accuracy_score, f1_score




## 2. Shared Utilities (Unified Interface)

In [ ]:
# Fix random seeds for reproducible experiments.
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Automatically locate the project root (must contain `dataset/archive` and `outputs`).
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'dataset' / 'archive').exists() and (p / 'outputs').exists():
            return p
    raise FileNotFoundError(' Graduation Thesis ')


class ResizeLongestSidePadSquare:
    def __init__(self, size: int, fill: int = 0):
        self.size = int(size)
        self.fill = int(fill)

    def __call__(self, img: Image.Image):
        w, h = img.size
        scale = self.size / max(w, h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        img = TF.resize(img, [nh, nw], interpolation=transforms.InterpolationMode.BILINEAR)
        pw, ph = self.size - nw, self.size - nh
        left, top = pw // 2, ph // 2
        right, bottom = pw - left, ph - top
        return TF.pad(img, [left, top, right, bottom], fill=self.fill)


class ManifestDataset(Dataset):
    # Build samples from the split manifest and return `(image, label, rel_path)`.
    def __init__(self, dataset_root: Path, df: pd.DataFrame, class_to_idx: dict, transform=None):
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.samples = []
        for _, r in df.iterrows():
            rel = str(r['path'])
            cls = str(r['class'])
            if cls not in self.class_to_idx:
                continue
            p = self.dataset_root / rel
            if p.exists():
                self.samples.append((p, self.class_to_idx[cls], rel))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, y, rel = self.samples[i]
        x = Image.open(p).convert('RGB')
        if self.transform is not None:
            x = self.transform(x)
        return x, y, rel


def build_baseline(model_key: str, num_classes: int, use_pretrained: bool = False):
    key = model_key.lower()
    if key == 'resnet18':
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.resnet18(weights=w)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if key == 'vgg16_bn':
        w = torchvision.models.VGG16_BN_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.vgg16_bn(weights=w)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    if key == 'vit_b_16':
        w = torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.vit_b_16(weights=w)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
        return m
    if key == 'swin_t':
        w = torchvision.models.Swin_T_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.swin_t(weights=w)
        m.head = nn.Linear(m.head.in_features, num_classes)
        return m
    raise ValueError(f' baseline: {model_key}')


def load_baseline(project_root: Path, model_key: str, num_classes: int, device: torch.device):
    ckpt = project_root / 'outputs' / 'baseline' / model_key / 'best.pt'
    if not ckpt.exists():
        raise FileNotFoundError(f'baseline checkpoint : {ckpt}')
    model = build_baseline(model_key, num_classes=num_classes, use_pretrained=False).to(device)
    payload = torch.load(ckpt, map_location=device)
    model.load_state_dict(payload['model_state_dict'])
    model.eval()
    return model, ckpt


def normalize_map(m: np.ndarray) -> np.ndarray:
    m = m.astype(np.float32)
    m -= m.min()
    d = m.max() + 1e-8
    return m / d


def topk_mask(score_map: np.ndarray, ratio: float = 0.1) -> np.ndarray:
    h, w = score_map.shape
    flat = score_map.reshape(-1)
    k = max(1, int(round(len(flat) * ratio)))
    idx = np.argpartition(flat, -k)[-k:]
    mask = np.zeros_like(flat, dtype=np.float32)
    mask[idx] = 1.0
    return mask.reshape(h, w)


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return float(inter / (union + 1e-8))


def deletion_aopc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5) -> float:
    # Deletion-AOPC: progressively remove high-importance regions and measure the drop in target confidence.
    with torch.no_grad():
        p0 = torch.softmax(model(x), dim=1)[0, cls_idx].item()

    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    x_work = x.clone()
    drops = []
    total = h * w

    for s in range(1, steps + 1):
        cut = int(total * s / steps)
        idx = order[:cut]
        m = np.ones(total, dtype=np.float32)
        m[idx] = 0.0
        m = torch.from_numpy(m.reshape(1, 1, h, w)).to(x.device)
        if x_work.shape[-2:] != (h, w):
            m = torch.nn.functional.interpolate(m, size=x_work.shape[-2:], mode='nearest')
        x_mask = x * m
        with torch.no_grad():
            p = torch.softmax(model(x_mask), dim=1)[0, cls_idx].item()
        drops.append(p0 - p)

    return float(np.mean(drops))


def insertion_auc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5) -> float:
    # Insertion-AUC: progressively insert high-importance regions and compute the area under the confidence curve.
    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    total = h * w
    xs = []
    ys = []

    for s in range(1, steps + 1):
        keep = int(total * s / steps)
        idx = order[:keep]
        m = np.zeros(total, dtype=np.float32)
        m[idx] = 1.0
        m = torch.from_numpy(m.reshape(1, 1, h, w)).to(x.device)
        if x.shape[-2:] != (h, w):
            m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')
        x_ins = x * m
        with torch.no_grad():
            p = torch.softmax(model(x_ins), dim=1)[0, cls_idx].item()
        xs.append(s / steps)
        ys.append(p)

    return float(np.trapezoid(ys, xs))




def comprehensiveness(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.1) -> float:
    # Faithfulness: drop in confidence after removing important regions; higher is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    with torch.no_grad():
        p_full = torch.softmax(model(x), dim=1)[0, cls_idx].item()
        p_drop = torch.softmax(model(x * (1.0 - m)), dim=1)[0, cls_idx].item()
    return float(p_full - p_drop)


def sufficiency(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.1) -> float:
    # Faithfulness: confidence loss when only important regions are kept; lower is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    with torch.no_grad():
        p_full = torch.softmax(model(x), dim=1)[0, cls_idx].item()
        p_keep = torch.softmax(model(x * m), dim=1)[0, cls_idx].item()
    return float(p_full - p_keep)


def selected_feature_ratio(score_map: np.ndarray, threshold: float = 0.5) -> float:
    # Complexity: selected-feature ratio; lower means sparser explanations.
    m = normalize_map(score_map)
    return float((m >= threshold).mean())


def sparsity_from_ratio(ratio: float) -> float:
    # Complexity: sparsity = 1 - selected-feature ratio; higher means sparser explanations.
    return float(1.0 - ratio)



def save_explanation_visual(x: torch.Tensor, score_map: np.ndarray, out_png: Path, mean, std):
    # Save a visualization that overlays the explanation heatmap on the input image.
    img = x.detach().cpu()[0].permute(1, 2, 0).numpy()
    mean = np.array(mean, dtype=np.float32).reshape(1, 1, 3)
    std = np.array(std, dtype=np.float32).reshape(1, 1, 3)
    img = np.clip(img * std + mean, 0.0, 1.0)
    h, w = img.shape[:2]

    hm = normalize_map(score_map)
    hm = np.array(Image.fromarray((hm * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)).astype(np.float32) / 255.0

    heat = np.zeros((h, w, 3), dtype=np.float32)
    heat[..., 0] = hm
    heat[..., 2] = 1.0 - hm

    overlay = np.clip(0.65 * img + 0.35 * heat, 0.0, 1.0)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((overlay * 255).astype(np.uint8)).save(out_png)

def stability_iou(explain_fn, model, x: torch.Tensor, cls_idx: int, top_ratio=0.1, noise_std=0.03) -> float:
    base_map, _ = explain_fn(model, x, cls_idx)
    n = torch.clamp(x + torch.randn_like(x) * noise_std, -5, 5)
    new_map, _ = explain_fn(model, n, cls_idx)
    a = topk_mask(base_map, top_ratio)
    b = topk_mask(new_map, top_ratio)
    return iou(a, b)





## 3. Global Configuration

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_ROOT = PROJECT_ROOT / 'dataset'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'data_prep'
SPLIT_ALL_PATH = SPLIT_DIR / 'split_all.csv'
PREPROCESS_YAML = SPLIT_DIR / 'preprocess_config.yaml'
PREPROCESS_JSON = SPLIT_DIR / 'preprocess_config.json'

# Default baseline (can be changed to `resnet18` / `vgg16_bn` / `vit_b_16` / `swin_t`).
BASELINE_KEY = 'swin_t'
MAX_SAMPLES = 100
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)

if PREPROCESS_YAML.exists():
    try:
        import yaml
        preprocess_cfg = yaml.safe_load(PREPROCESS_YAML.read_text(encoding='utf-8'))
    except Exception:
        preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8')) if PREPROCESS_JSON.exists() else {}
elif PREPROCESS_JSON.exists():
    preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8'))
else:
    preprocess_cfg = {}

IMG_SIZE = int(preprocess_cfg.get('img_size', 224))
PAD_FILL = int(preprocess_cfg.get('pad_fill', 0))
NORM_MEAN = tuple(preprocess_cfg.get('normalize', {}).get('mean', [0.485, 0.456, 0.406]))
NORM_STD = tuple(preprocess_cfg.get('normalize', {}).get('std', [0.229, 0.224, 0.225]))

split_df = pd.read_csv(SPLIT_ALL_PATH)
classes = sorted(split_df['class'].dropna().unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

test_df = split_df[split_df['split'] == 'test'].copy().reset_index(drop=True)
if MAX_SAMPLES > 0:
    test_df = test_df.iloc[:MAX_SAMPLES].copy().reset_index(drop=True)

eval_tfms = transforms.Compose([
    ResizeLongestSidePadSquare(IMG_SIZE, fill=PAD_FILL),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

test_ds = ManifestDataset(DATASET_ROOT, test_df, class_to_idx, transform=eval_tfms)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

baseline_model, baseline_ckpt = load_baseline(PROJECT_ROOT, BASELINE_KEY, len(classes), DEVICE)
print('[INFO] baseline =', BASELINE_KEY)
print('[INFO] ckpt =', baseline_ckpt)
print('[INFO] test samples =', len(test_ds))




## 4. Method Implementation

In [ ]:
METHOD_KEY = 'lime'
VARIANT_KEY = 'post-hoc'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'xai' / 'post-hoc' / METHOD_KEY / BASELINE_KEY

# Heavier LIME settings (adjust if runtime is too long)
LIME_NUM_SAMPLES = 1000
LIME_N_SEGMENTS = 120
LIME_COMPACTNESS = 12.0
LIME_SIGMA = 1.0
LIME_BATCH_SIZE = 64
LIME_POSITIVE_ONLY = True

def explain_fn(model, x: torch.Tensor, cls_idx: int):
    t0 = time.time()

    try:
        from lime import lime_image  # type: ignore
        from skimage.segmentation import slic  # type: ignore

        # 1) convert normalized tensor -> [0,1] RGB numpy
        x_np = x.detach().cpu()[0].permute(1, 2, 0).numpy().astype(np.float32)
        mean = np.array(NORM_MEAN, dtype=np.float32).reshape(1, 1, 3)
        std = np.array(NORM_STD, dtype=np.float32).reshape(1, 1, 3)
        img = np.clip(x_np * std + mean, 0.0, 1.0).astype(np.float32)

        mean_t = torch.tensor(NORM_MEAN, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
        std_t = torch.tensor(NORM_STD, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)

        # 2) classifier_fn for LIME perturbations
        def classifier_fn(images: np.ndarray):
            arr = np.asarray(images, dtype=np.float32)
            if arr.ndim == 3:
                arr = arr[None, ...]

            probs_all = []
            for s in range(0, arr.shape[0], LIME_BATCH_SIZE):
                b = arr[s:s + LIME_BATCH_SIZE]
                bt = torch.from_numpy(b).permute(0, 3, 1, 2).to(DEVICE)
                bt = (bt - mean_t) / std_t
                with torch.no_grad():
                    p = torch.softmax(model(bt), dim=1)
                probs_all.append(p.detach().cpu().numpy())
            return np.concatenate(probs_all, axis=0)

        # 3) segmentation + LIME explanation
        def segmentation_fn(image):
            return slic(
                image,
                n_segments=LIME_N_SEGMENTS,
                compactness=LIME_COMPACTNESS,
                sigma=LIME_SIGMA,
                start_label=0,
            )

        explainer = lime_image.LimeImageExplainer(random_state=SEED)
        import inspect
        kwargs = dict(
            classifier_fn=classifier_fn,
            labels=(int(cls_idx),),
            top_labels=None,
            hide_color=0.0,
            num_samples=int(LIME_NUM_SAMPLES),
            segmentation_fn=segmentation_fn,
        )
        if 'progress_bar' in inspect.signature(explainer.explain_instance).parameters:
            kwargs['progress_bar'] = False
        exp = explainer.explain_instance(img, **kwargs)

        seg = exp.segments
        local = dict(exp.local_exp.get(int(cls_idx), []))
        if len(local) == 0:
            g = np.zeros_like(seg, dtype=np.float32)
            return g, time.time() - t0

        g = np.zeros_like(seg, dtype=np.float32)
        for sid, w in local.items():
            v = float(w)
            if LIME_POSITIVE_ONLY:
                v = max(0.0, v)
            else:
                v = abs(v)
            g[seg == int(sid)] = v

        # if all-positive map is empty, fallback to absolute weights
        if float(g.max()) <= 1e-12:
            for sid, w in local.items():
                g[seg == int(sid)] = abs(float(w))

        g = normalize_map(g)
        return g, time.time() - t0

    except Exception as e:
        if not getattr(explain_fn, '_warned', False):
            print(f'[WARN] LIME backend unavailable, fallback to gradient saliency: {e}')
            explain_fn._warned = True
        x2 = x.clone().detach().requires_grad_(True)
        logits = model(x2)
        score = logits[0, cls_idx]
        model.zero_grad(set_to_none=True)
        score.backward()
        g = x2.grad.detach().abs().mean(dim=1)[0].cpu().numpy()
        g = normalize_map(g)
        return g, time.time() - t0



## 5. Unified Quantitative Evaluation and Outputs

In [ ]:
VIS_SAVE_N = 20
VIS_DIR = OUT_DIR / 'visuals'
VIS_DIR.mkdir(parents=True, exist_ok=True)

records = []
for i, (x, y, rel) in enumerate(test_loader):
    sample_t0 = time.time()
    x = x.to(DEVICE)
    y = y.to(DEVICE)

    with torch.no_grad():
        logits = baseline_model(x)
        pred = int(logits.argmax(1).item())

    score_map, t_explain = explain_fn(baseline_model, x, pred)
    if i < VIS_SAVE_N:
        stem = Path(rel[0]).stem
        vis_path = VIS_DIR / f"{i:04d}_{stem}.png"
        save_explanation_visual(x, score_map, vis_path, NORM_MEAN, NORM_STD)

    aopc = deletion_aopc(baseline_model, x, pred, score_map)
    auci = insertion_auc(baseline_model, x, pred, score_map)
    siou = stability_iou(explain_fn, baseline_model, x, pred)
    comp = comprehensiveness(baseline_model, x, pred, score_map)
    suff = sufficiency(baseline_model, x, pred, score_map)
    sfr = selected_feature_ratio(score_map, threshold=0.5)
    spr = sparsity_from_ratio(sfr)
    sample_elapsed = time.time() - sample_t0
    print(f"[{i+1:03d}/{len(test_loader):03d}] explain done | explain={t_explain:.3f}s | total={sample_elapsed:.3f}s | aopc={aopc:.4f} | auc={auci:.4f}")

    records.append({
        'idx': int(i),
        'path': rel[0],
        'y_true': int(y.item()),
        'y_pred': int(pred),
        'explain_time_sec': float(t_explain),
        'aopc_deletion': float(aopc),
        'auc_insertion': float(auci),
        'iou_top10': float(siou),
        'comprehensiveness': float(comp),
        'sufficiency': float(suff),
        'selected_feature_ratio': float(sfr),
        'sparsity': float(spr),
    })

res_df = pd.DataFrame(records)

summary = {
    'method': METHOD_KEY,
    'variant': VARIANT_KEY,
    'baseline_key': BASELINE_KEY,
    'num_samples': int(len(res_df)),
    'metrics': {
        'mean_explain_time_sec': float(res_df['explain_time_sec'].mean()) if len(res_df) else None,
        'median_explain_time_sec': float(res_df['explain_time_sec'].median()) if len(res_df) else None,
        'mean_aopc_deletion': float(res_df['aopc_deletion'].mean()) if len(res_df) else None,
        'mean_auc_insertion': float(res_df['auc_insertion'].mean()) if len(res_df) else None,
        'mean_iou_top10': float(res_df['iou_top10'].mean()) if len(res_df) else None,
        'mean_comprehensiveness': float(res_df['comprehensiveness'].mean()) if len(res_df) else None,
        'mean_sufficiency': float(res_df['sufficiency'].mean()) if len(res_df) else None,
        'mean_selected_feature_ratio': float(res_df['selected_feature_ratio'].mean()) if len(res_df) else None,
        'mean_sparsity': float(res_df['sparsity'].mean()) if len(res_df) else None,
    }
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
(res_df).to_csv(OUT_DIR / 'per_sample_metrics.csv', index=False, encoding='utf-8')
(OUT_DIR / 'quant_metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('[OK] saved:', OUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))



